In [1]:
library(tidyverse)
library(readxl)
library(here)
library(glue)
library(janitor)
i_am("ncrecords/import-provider-days.ipynb")

data_dir <- here("ncrecords/nctracks/institutional-days")



── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors
here() starts at /Users/sarah/Github/nc-homesdisabled

Attaching package: ‘janitor’

The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test

here() starts at /Users/sarah/Github/nc-homesdisabled


Although North Carolina has historical data going back to 2014, it's useless until 2023. That's because, until then, it only included payments for FFS, not the managed care entities. They've been transitioning to that for a long time and by about 2020 or so, there were virtually no FFS providers left except the state ones. 

This means only the following files are useful for us: 

```
SFY 2023 Service Provider Medicaid Institutional Days.csv" 
SFY 2024 Service Provider Medicaid Institutional Days.xlsx"
SFY 2025 Service Provider Medicaid Institutional Days.csv" 

 ```

 The csvs have unformatted amounts, the xlsx has formatted amounts. 


 These are essentially identical, but there are some name changes in them. I'll deal with them one at a time, since there are only three:

In [2]:
use_files <- list.files(path = here("ncrecords/nctracks/institutional-days"), pattern="SFY 202[345]")




Find the first row of data, for csv and Excel files. We can use this if we want to go backwards. 


In [3]:
get_first_row <- function ( filename,  marker) {
    if ( str_ends( filename, "csv") ) {
        peek <- read_csv( filename, 
                            col_names=FALSE,
                            n_max=60,
                            col_types=cols(.default="c"), 
                            name_repair = "unique_quiet", 
                            show_col_types = FALSE) 
    } else if (str_ends ( filename, "xls.")) {
          peek <- read_excel(filename, sheet = 1, col_names = FALSE,
                      n_max = 60, col_types = "text", .name_repair="unique_quiet")  
    }

    first_row <- peek |> 
        rowid_to_column() |> 
        mutate  ( found = if_any ( 1:5, \(x) str_starts(x, "Billing NPI")) ) |> 
        filter ( found == TRUE) 

    first_row |>  pull ( rowid) 

}



Test the first rows. These are a mix of csv and excel, so i need to either do them separately or set something up to test for it 

In [4]:
# get_first_row( glue("{data_dir}/{use_files[1]}"))

map_int ( glue("{data_dir}/{use_files}") , get_first_row) 

[1] 22 22 21

Might as well set something up to test since I may need to add some. 

In [5]:



get_orig_data <- function ( filename) {

  # use the function to find the first row of datat
  
  start_after <- get_first_row ( filename, "Billing NPI") -1    
  # what fiscal year is this? 
  sfy <-  str_extract(filename, "20\\d\\d")

  # what file type is this? 
  ftype <- tools::file_ext( filename) 

  # common things: Use the column names, set as na nything with "#" or "$#" as the value. These are redacted
  # for csv, use read_csv
  if ( ftype == "csv") {

    df <- read_csv( filename, skip =  start_after ,
                            col_names=TRUE,
                            col_types=cols(.default="c"), 
                            name_repair = "unique_quiet", 
                            show_col_types = FALSE, 
                            na = c("#", "$#", "")
                          )  |> 
      
    # there's a phantom column at the end
    select ( -c(  starts_with ("...") ) )


  } else if ( ftype == "xlsx") {

    # for excel use read_excel
    df <- read_excel( filename, 
                              skip = start_after, 
                              col_names = TRUE, 
                              col_types="text",
                               na = c("#", "$#", ""),
                               .name_repair="unique_quiet"
                              ) 

  }
 
   # now fix some things: column names, filter for ICFs  
  df |>  mutate ( fiscal_year = sfy, orig_fname = basename(filename)) |> 
    clean_names() |> 
    # don't filter at this point - get the whole thing, and check later on
    # filter (rendering_taxonomy_code == "315P00000X" ) |>
    # convert numbers to numbers, and rename them so I can check later on.
    mutate ( across ( number_of_recipients:amount_paid , \(x) parse_number(x), .names= "num_{.col}")  )

}







In [6]:
list_of_dfs <- map (  glue("{data_dir}/{use_files}"), \(x) get_orig_data(x) ) 
all_payments <- list_rbind( list_of_dfs)  

In [11]:
# save this

saveRDS(all_payments, file = "nctracks/icf_payments_3year.RDS")

## Test values

Just get a sense of how much they are spending


What are the taxonomy codes used ? Make sure there's only one for ICFs

In [10]:
count ( all_payments, billing_provider_taxonomy_code, billing_provider_type,
   billing_provider_classification, rendering_taxonomy_code, rendering_provider_type, rendering_provider_classification)

# A tibble: 29 × 7
   billing_provider_taxonomy_code billing_provider_type             billing_provider_cla…¹ rendering_taxonomy_c…² rendering_provider_t…³
   <chr>                          <chr>                             <chr>                  <chr>                  <chr>                 
 1 193200000X                     Group                             Multi-Specialty        193200000X             Group                 
 2 193400000X                     Group                             Single Specialty       193400000X             Group                 
 3 207R00000X                     Allopathic & Osteopathic Physici… Internal Medicine      207R00000X             Allopathic & Osteopat…
 4 208200000X                     Allopathic & Osteopathic Physici… Plastic Surgery        208200000X             Allopathic & Osteopat…
 5 237600000X                     Speech, Language and Hearing Ser… Audiologist-Hearing A… 237600000X             Speech, Language and …
 6 251B00000X         

In [9]:
library(gt)

name_cols <- c("billing_provider_name", "rendering_provider_name" ) 

all_payments <- 
  all_payments |> 
  mutate(is_rha = if_any(
      any_of(c("billing_provider_name", "rendering_provider_name")),
            \(x) str_detect(coalesce(x, ""), "^RHA|Howell")
      ) |> as.character()  |> recode("TRUE" = "RHA", "FALSE" = "Not RHA")
    )
  

glimpse(all_payments)


Rows: 3,515
Columns: 31
$ payer                             <chr> "Medicaid Direct", "Medicaid Direct", "Medicaid Direct", "Alliance Health", "Trillium Health Resources", "Alliance Health", "Partners …
$ billing_npi_atypical              <chr> "1699811125", "1871656504", "1902083975", NA, "1003021874", "1003936238", "1003936238", "1003951542", "1003951542", "1003957556", "100…
$ billing_provider_name             <chr> "MURDOCH DEVELOPMENTAL CENTER", "RHA HEALTH SERVICES NC LLC", "HILLTOP HOME INC", NA, "LIFETIME RESOURCES INC", "EDUCARE COMMUNITY LIV…
$ billing_provider_dba_name         <chr> "MURDOCH DEVELOPMENTAL CENTER", NA, "HILLTOP HOME", NA, NA, NA, NA, NA, NA, "MONARCH", "MONARCH", "MONARCH", "MONARCH", NA, NA, NA, NA…
$ billing_address                   <chr> "1600 E C ST   BUTNER NC", "500 SEAN DR   GREENVILLE NC", "2820 KIDD RD   RALEIGH NC", NA, "220 DORCHESTER PL   WILMINGTON NC", "2823 …
$ billing_county_code               <chr> "039", "074", "092", NA, "065", "060", "060"

This is just playing around with table-making packages. The gtsummary doesn't have robust formatting, and , importantly, doesn't have column sums. 

In [10]:
library(gtsummary)

all_payments |> 
  filter ( !is.na(num_amount_paid)) |> 
  tbl_continuous( 
    variable = num_amount_paid, 
    by = is_rha, 
    include = fiscal_year, 
    statistic = ~"{sum} ({N_obs})"
  )  |> 
  add_overall() |> 
  as_gt()
  

<R HTML widget>

I would think you can play with this a lot. There are also bootstrap options. Ipynb files, though, won't show any of this. 


In [12]:
#| df_print: paged

library(tables)
library(kableExtra)
 
table_options(CSS =
"<style>
#ID .center { 
  text-align:right;
  background-color: aliceblue;
}
</style>", doCSS = TRUE)





Attaching package: ‘kableExtra’

The following object is masked from ‘package:dplyr’:

    group_rows



In [15]:
sum_na <- function (x) sum(x, na.rm=TRUE) 




tab <-
tabular(
  (Heading("Fiscal Year") * Factor(fiscal_year) + Heading("Total") * 1) ~ 
  (Heading() * Factor(is_rha)  + Heading("All providers") * 1) *  
  (Heading(" $ ") * num_amount_paid * Heading () * sum_na * Format(scientific=FALSE, big.mark=",") + 
   Heading(" # ") * num_amount_paid * Heading () * length ), 
  data = all_payments
)

# toHTML(tab)
tab


                                                                     
             Not RHA            RHA                All providers     
 fiscal_year  $             #    $             #    $             #  
 2023          682,652,021  814   255,964,009  388   938,616,030 1202
 2024          443,752,277  693   102,428,130  348   546,180,407 1041
 2025          545,226,817  860   172,650,962  412   717,877,780 1272
 Total       1,671,631,115 2367   531,043,102 1148 2,202,674,216 3515

NOTE: "RHA" is defined as any biller or provider whose name starts with "RHA" or includes "Howell" 
I'll come back and do this by NPI later on.  This only includes providers that are listed as ICFs, not group homes or day side providers. 


In [18]:
nrow ( all_payments) 
all_payments |> 
  # get rid of the missing amounts 
  filter ( ! is.na( amount_paid)) |> 
  get_dupes ( billing_npi_atypical, rendering_npi_atypical,  fiscal_year)  |> arrange ( billing_npi_atypical)

# A tibble: 3,185 × 32
   billing_npi_atypical rendering_npi_atypical fiscal_year dupe_count payer   billing_provider_name billing_provider_dba…¹ billing_address billing_county_code billing_county_name
   <chr>                <chr>                  <chr>            <int> <chr>   <chr>                 <chr>                  <chr>           <chr>               <chr>              
 1 1003021874           1003021874             2025                 2 TP-Tri… LIFETIME RESOURCES I… NA                     220 DORCHESTER… 065                 New Hanover        
 2 1003021874           1003021874             2025                 2 Trilli… LIFETIME RESOURCES I… NA                     220 DORCHESTER… 065                 New Hanover        
 3 1003936238           1003936238             2024                 3 Partne… EDUCARE COMMUNITY LI… NA                     NA              NA                  Unknown            
 4 1003936238           1003936238             2024                 3 Partne… EDUC